# Multilingual MiniLM — Tagalog NSFW Text Classifier

> **Model-size note (read this first).**
>
> This notebook fine-tunes **`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`**.
> Unlike TinyBERT (English-only base), this MiniLM is **multilingually pre-trained on 50+
> languages including Filipino/Tagalog**, so it is a *language-fair* peer of the Tagalog-native
> models (DOST-RoBERTa-tl, DistilBERT-Tagalog) and the from-scratch LSTM/BiLSTM.
>
> The trade-off is **size**: MiniLM-L12 carries a ~250k-token multilingual vocabulary and is the
> **largest base in the study (~118M parameters)** — an order of magnitude bigger than TinyBERT or
> the RNNs. Its on-device artifact is therefore the heaviest, which is exactly why it is kept here
> as a **multilingual-transformer benchmark**: it shows the accuracy ceiling a large multilingual
> base reaches, against which the compact models are judged on the accuracy-vs-size curve.
>
> To keep the comparison fair, this notebook follows the **same protocol** as the other six models:
> a **70/15/15 stratified split**, **macro-averaged** metrics, a **validation-set-calibrated
> decision threshold**, and TFLite export through the shared **litert-torch (ai-edge-torch)** route
> (with a data-free **dynamic-range INT8** build) so on-device sizes are directly comparable.

In [1]:
!pip install onnxscript optimum[onnxruntime] onnxconverter-common litert-torch torchao

INFO: pip is looking at multiple versions of ai-edge-litert[model-utils] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of onnx to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of optimum-onnx[onnxruntime] to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.8/575.8 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.3/419.3 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/1

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import time
warnings.filterwarnings('ignore')

In [3]:
try:
    from optimum.onnxruntime import ORTModelForSequenceClassification
    from optimum.onnxruntime.configuration import OptimizationConfig
    ONNX_AVAILABLE = True
except ImportError:
    print("Optimum ONNX Runtime not available, ONNX export will be limited")
    ONNX_AVAILABLE = False

Multiple distributions found for package optimum. Picked distribution: optimum
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [4]:
# Set random seeds for reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [5]:
# Multilingual MiniLM Classifier Model
# Base: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
#   - 12 transformer layers, hidden size 384, ~250k multilingual WordPiece/SP vocab
#   - multilingually pre-trained (50+ languages incl. Filipino/Tagalog)
# A custom [CLS]-pooling head is used (same head design as the TinyBERT baseline),
# so the architecture stays MiniLM's own while the training/eval/export wrapper
# matches the rest of the study.
class MiniLMClassifier(nn.Module):
    def __init__(self, bert_model_name, num_labels=2, dropout=0.3):
        super(MiniLMClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # Use [CLS] token representation
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

In [6]:
# Custom Dataset Class
class NSFWDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [7]:
# Training Function
def train_epoch(model, dataloader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    predictions, true_labels = [], []

    progress_bar = tqdm(dataloader, desc="Training")
    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        preds = torch.argmax(logits, dim=1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

        progress_bar.set_postfix({'loss': loss.item()})

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)

    return avg_loss, accuracy

In [8]:
# Evaluation Function
# Returns class probabilities as well, so the validation set can be used to
# calibrate a decision threshold (matching the RoBERTa / DistilBERT / LSTM / BiLSTM protocol).
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    predictions, true_labels, probabilities = [], [], []

    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc="Evaluating")
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
            probabilities.extend(probs.cpu().numpy())

    avg_loss = total_loss / len(dataloader)

    return avg_loss, predictions, true_labels, np.array(probabilities)

In [9]:
# Validation-set threshold calibration (mirrors RoBERTa / DistilBERT / LSTM / BiLSTM)
def find_optimal_threshold(probs, y_true):
    """Pick the probability threshold that maximises macro-F1 on the validation set.

    Searches a sane band and only deviates from the neutral 0.5 if the validation gain
    exceeds 0.01, so the operating point matches the policy used by the other models
    and the test numbers stay directly comparable.
    """
    y_true = np.asarray(y_true)
    pos = np.asarray(probs)[:, 1]

    thresholds = np.arange(0.30, 0.71, 0.01)
    f1s = [f1_score(y_true, (pos >= t).astype(int), average='macro', zero_division=0) for t in thresholds]
    best_idx = int(np.argmax(f1s))
    tuned_t, tuned_f1 = float(thresholds[best_idx]), f1s[best_idx]

    f1_half = f1_score(y_true, (pos >= 0.5).astype(int), average='macro', zero_division=0)
    if tuned_f1 - f1_half > 0.01:
        best_t, best_f1 = tuned_t, tuned_f1
    else:
        best_t, best_f1 = 0.5, f1_half

    print(f" Optimal Threshold: {best_t:.2f} (tuned {tuned_t:.2f}->F1 {tuned_f1:.4f} vs 0.50->F1 {f1_half:.4f})")
    print(f" Best Validation Macro-F1: {best_f1:.4f}")
    return best_t

In [10]:
# Plot Confusion Matrix
def plot_confusion_matrix(cm, save_path='confusion_matrix_minilm.png'):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['SFW', 'NSFW'],
                yticklabels=['SFW', 'NSFW'],
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.title('Confusion Matrix - Multilingual MiniLM NSFW Classifier', fontsize=14, fontweight='bold')

    # Add percentage annotations
    total = cm.sum()
    for i in range(2):
        for j in range(2):
            percentage = (cm[i, j] / total) * 100
            plt.text(j + 0.5, i + 0.7, f'({percentage:.1f}%)',
                    ha='center', va='center', fontsize=10, color='gray')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f" Confusion matrix saved to {save_path}")
    plt.close()

In [11]:
# Plot Training History
def plot_training_history(train_losses, val_losses, train_accs, val_accs,
                         save_path='training_history_minilm.png'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    epochs = range(1, len(train_losses) + 1)

    # Loss plot
    ax1.plot(epochs, train_losses, 'b-o', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-o', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Loss', fontsize=12, fontweight='bold')
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # Accuracy plot
    ax2.plot(epochs, train_accs, 'b-o', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accs, 'r-o', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f" Training history saved to {save_path}")
    plt.close()

In [12]:
# Inference Function
def predict(text, model, tokenizer, device, max_length=128):
    model.eval()

    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    start_time = time.time()
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1)
    inference_time = (time.time() - start_time) * 1000  # Convert to milliseconds

    return pred.item(), probs.cpu().numpy()[0], inference_time

In [13]:
# Batch Inference Function
def batch_predict(texts, model, tokenizer, device, max_length=128, batch_size=32):
    model.eval()
    all_predictions = []
    all_probabilities = []

    dataset = NSFWDataset(texts, [0] * len(texts), tokenizer, max_length)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Batch Inference"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_predictions.extend(preds.cpu().numpy())
            all_probabilities.extend(probs.cpu().numpy())

    return all_predictions, all_probabilities

In [14]:
# Export to ONNX Function
def export_to_onnx(model, tokenizer, onnx_path, device):
    import os
    print("Exporting Multilingual MiniLM NSFW Classifier to ONNX format...")

    try:
        # Create dummy input with sample Tagalog text (same max_length as training: 128)
        dummy_input = tokenizer(
            "Magandang umaga sa lahat",
            return_tensors="pt",
            max_length=128,
            padding="max_length",
            truncation=True
        )

        input_ids = dummy_input["input_ids"].to(device)
        attention_mask = dummy_input["attention_mask"].to(device)

        model.eval()

        os.makedirs(os.path.dirname(onnx_path), exist_ok=True)
        torch.onnx.export(
            model,
            (input_ids, attention_mask),
            onnx_path,
            export_params=True,
            opset_version=18,
            do_constant_folding=True,
            input_names=['input_ids', 'attention_mask'],
            output_names=['logits'],
            dynamic_axes={
                'input_ids': {0: 'batch_size', 1: 'sequence'},
                'attention_mask': {0: 'batch_size', 1: 'sequence'},
                'logits': {0: 'batch_size'}
            }
        )

        print(f"ONNX model exported to: {onnx_path}")

        # Report a FAIR size: include the external weight sidecar (.onnx.data) if
        # the exporter spilled the weights into one, otherwise the headline number
        # is misleadingly small (the graph alone is < 1 MB).
        total_bytes = os.path.getsize(onnx_path)
        sidecar = onnx_path + ".data"
        if os.path.exists(sidecar):
            total_bytes += os.path.getsize(sidecar)
            print(f"ONNX external weights: {sidecar}")
        print(f"ONNX model size (graph + weights): {total_bytes / (1024 * 1024):.2f} MB")

        return True

    except Exception as e:
        print(f"ONNX export failed: {e}")
        import traceback
        traceback.print_exc()
        return False

In [15]:
# Export to TFLite via litert-torch (ai-edge-torch) -- same route as the other transformers
def export_to_tflite_ai_edge(model, model_name, output_dir, max_length=128):
    """Convert the trained MiniLM classifier to TFLite via litert-torch (LiteRT).

    transformers v5 dropped native TF, so the old TFAutoModel(from_pt=True) -> TFLite
    route no longer works (and previously produced a bloated ~224 MB FP16 file).
    litert-torch (formerly 'ai-edge-torch') converts PyTorch -> LiteRT directly and
    supports a data-free dynamic-range INT8 recipe. This matches the DistilBERT /
    DOST_RoBERTa / TinyBERT export path, so every transformer in the study ships
    comparable artifacts and we can report a fair on-device size.

    Emits:
      - minilm_nsfw_float32.tflite : reference build (largest)
      - minilm_nsfw_int8.tflite    : dynamic-range INT8 weights (~1/4 size)
    Both import via Android Studio's ML Model Binding.
    """
    import importlib, os
    print("Exporting Multilingual MiniLM NSFW Classifier to TFLite (litert-torch)...")

    # 'ai-edge-torch' was renamed to 'litert-torch'; the old name is now a stub
    # without .convert. Use whichever module actually provides the API.
    aet = None
    for modname in ('litert_torch', 'ai_edge_torch'):
        try:
            m = importlib.import_module(modname)
        except Exception:
            continue
        if hasattr(m, 'convert'):
            aet = m
            break
    if aet is None:
        print("No working litert_torch/ai_edge_torch (.convert) found; skipping TFLite export.")
        return False
    pkg = aet.__name__
    print(f"  Using '{pkg}' for TFLite conversion.")

    try:
        # Rebuild a clean CPU eval copy with eager attention. eager avoids SDPA's
        # data-dependent branches, which torch.export cannot trace. The custom
        # MiniLMClassifier.forward already returns the logits tensor directly,
        # so no logits-unwrapping wrapper is needed.
        export_model = MiniLMClassifier(model_name, num_labels=2)
        export_model.bert = AutoModel.from_pretrained(model_name, attn_implementation="eager")
        export_model.load_state_dict(model.state_dict())
        export_model.eval().to("cpu")

        tokenizer = AutoTokenizer.from_pretrained(model_name)

        # Fixed [1, max_length] sample -> the .tflite takes a fixed-length input.
        enc = tokenizer(
            "Ito ay isang sample na Tagalog text", return_tensors="pt",
            max_length=max_length, padding="max_length", truncation=True
        )
        sample = (enc["input_ids"], enc["attention_mask"])

        os.makedirs(output_dir, exist_ok=True)
        ok = False

        # --- Float32 LiteRT model (reliable baseline) ---
        try:
            print(f"  Converting PyTorch -> TFLite (float32) via {pkg}...")
            edge = aet.convert(export_model, sample)
            f32 = os.path.join(output_dir, "minilm_nsfw_float32.tflite")
            edge.export(f32)
            print(f"  TFLite (float32) saved to {f32} ({os.path.getsize(f32) / (1024 * 1024):.2f} MB)")
            ok = True
        except Exception as e:
            print(f"  {pkg} float32 conversion failed: {e}")
            import traceback
            traceback.print_exc()

        # --- Dynamic-range INT8 (data-free; ~1/4 size) ---
        try:
            pt2e = importlib.import_module(f"{pkg}.quantize.pt2e_quantizer")
            qcfg = importlib.import_module(f"{pkg}.quantize.quant_config")
            PT2EQuantizer = pt2e.PT2EQuantizer
            get_symmetric_quantization_config = pt2e.get_symmetric_quantization_config
            QuantConfig = qcfg.QuantConfig

            # prepare_pt2e/convert_pt2e moved out of torch.ao into torchao in newer torch.
            prepare_pt2e = convert_pt2e = None
            for modpath in (
                'torchao.quantization.pt2e.quantize_pt2e',
                'torch.ao.quantization.quantize_pt2e',
                'torch.ao.quantization.pt2e.quantize_pt2e',
            ):
                try:
                    qmod = importlib.import_module(modpath)
                    prepare_pt2e, convert_pt2e = qmod.prepare_pt2e, qmod.convert_pt2e
                    break
                except Exception:
                    continue
            if prepare_pt2e is None:
                raise ImportError("prepare_pt2e/convert_pt2e not found (torch.ao or torchao)")

            print(f"  Converting PyTorch -> TFLite (dynamic-range INT8) via {pkg} PT2E...")
            quantizer = PT2EQuantizer().set_global(
                get_symmetric_quantization_config(is_per_channel=True, is_dynamic=True)
            )

            # Capture a pre-autograd graph (entry point name varies across torch versions).
            captured = None
            try:
                captured = torch.export.export_for_training(export_model, sample).module()
            except (AttributeError, TypeError):
                try:
                    from torch._export import capture_pre_autograd_graph
                    captured = capture_pre_autograd_graph(export_model, sample)
                except Exception:
                    captured = torch.export.export(export_model, sample).module()

            prepared = prepare_pt2e(captured, quantizer)
            converted = convert_pt2e(prepared)
            edge_q = aet.convert(
                converted, sample, quant_config=QuantConfig(pt2e_quantizer=quantizer)
            )
            i8 = os.path.join(output_dir, "minilm_nsfw_int8.tflite")
            edge_q.export(i8)
            print(f"  TFLite (INT8) saved to {i8} ({os.path.getsize(i8) / (1024 * 1024):.2f} MB)")
            ok = True
        except Exception as e:
            print(f"  {pkg} INT8 conversion failed: {e}")
            import traceback
            traceback.print_exc()

        if ok:
            print("  Import in Android Studio via: File > New > Other > TensorFlow Lite Model.")
        else:
            print(f"  {pkg} produced no TFLite files.")
        return ok

    except Exception as e:
        print(f"TFLite export failed: {e}")
        import traceback
        traceback.print_exc()
        return False

In [16]:
# Main Training Pipeline
def main(csv_path, epochs=5, batch_size=32, learning_rate=2e-5, max_length=128):
    import os
    import sys
    import time
    import json

    # Check if data exists
    if not os.path.exists(csv_path):
        print(f"Error: The file '{csv_path}' was not found.")
        print("Terminating program as cleaned_data.csv is required.")
        sys.exit("Dataset not found.")

    # Load data
    print(" Loading dataset.")
    df = pd.read_csv(csv_path)

    # Transform labels to binary if they are strings
    if df.iloc[:, 1].dtype == object:
        print("  Transforming text labels to binary...")
        label_map = {'safe': 0, 'nsfw': 1}
        df.iloc[:, 1] = df.iloc[:, 1].astype(str).str.lower().str.strip().map(label_map)

        if df.iloc[:, 1].isnull().any():
            nan_count = df.iloc[:, 1].isnull().sum()
            print(f"  Warning: Found {nan_count} rows with unknown labels. Dropping them.")
            df = df.dropna(subset=[df.columns[1]])

        df.iloc[:, 1] = df.iloc[:, 1].astype(int)

    # Assume first column is text, second column is label
    texts = df.iloc[:, 0].values
    labels = df.iloc[:, 1].values

    print(f"  Dataset loaded successfully!")
    print(f"  Total samples: {len(texts):,}")
    print(f"  Label distribution:")
    print(f"    - SFW (0): {(labels == 0).sum():,} ({(labels == 0).sum()/len(labels)*100:.1f}%)")
    print(f"    - NSFW (1): {(labels == 1).sum():,} ({(labels == 1).sum()/len(labels)*100:.1f}%)")

    # Split data 70/15/15 (stratified) -- same protocol as the other six models
    print("\n Splitting dataset (70/15/15, stratified).")
    X_train, X_temp, y_train, y_temp = train_test_split(
        texts, labels, test_size=0.3, random_state=42, stratify=labels
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    print(f"  Training set: {len(X_train):,} samples")
    print(f"  Validation set: {len(X_val):,} samples")
    print(f"  Test set: {len(X_test):,} samples")

    # Initialize tokenizer and model
    print("\n Initializing Multilingual MiniLM model.")
    # sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 (12 layers, hidden 384)
    model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = MiniLMClassifier(model_name, num_labels=2)

    # --- MULTILINGUAL-BASE NOTICE ------------------------------------------
    print("  NOTE: 'paraphrase-multilingual-MiniLM-L12-v2' is multilingually pre-trained")
    print("        on 50+ languages INCLUDING Filipino/Tagalog, so it is a language-fair")
    print("        peer of the Tagalog-native models. Its trade-off is SIZE: a ~250k-token")
    print("        multilingual vocabulary makes it the largest base in the study (~118M")
    print("        params). Treat it as the multilingual-transformer benchmark on the")
    print("        accuracy-vs-size curve.")
    # ------------------------------------------------------------------------

    # Setup device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  Device: {device}")

    # Model info
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    print(f"  Model size: ~{total_params * 4 / (1024**2):.1f} MB")

    model.to(device)

    # Create datasets and dataloaders
    print("\n Data loaders.")
    train_dataset = NSFWDataset(X_train, y_train, tokenizer, max_length)
    val_dataset = NSFWDataset(X_val, y_val, tokenizer, max_length)
    test_dataset = NSFWDataset(X_test, y_test, tokenizer, max_length)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    print(f" Data loaders ready!")

    # Setup optimizer, scheduler, and loss
    print("\n Configuring training parameters.")
    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )
    criterion = nn.CrossEntropyLoss()
    print(f"  Optimizer: AdamW (lr={learning_rate})")
    print(f"  Total training steps: {total_steps:,}")

    # Training loop
    print("\n Training model.")

    best_val_loss = float('inf')
    best_val_acc = 0.0
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    for epoch in range(epochs):
        print(f"\n EPOCH {epoch + 1}/{epochs} ")

        train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
        val_loss, val_preds, val_labels, _ = evaluate(model, val_loader, criterion, device)

        val_acc = accuracy_score(val_labels, val_preds)
        val_f1_macro = f1_score(val_labels, val_preds, average='macro', zero_division=0)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f" Training:   Loss {train_loss:.4f} | Accuracy {train_acc*100:.2f}%")
        print(f" Validation: Loss {val_loss:.4f} | Accuracy {val_acc*100:.2f}% | Macro-F1 {val_f1_macro:.4f}")

        # Save best model (by validation loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_acc': val_acc,
                'train_losses': train_losses,
                'val_losses': val_losses,
                'train_accs': train_accs,
                'val_accs': val_accs
            }, 'best_minilm_nsfw_model.pth')
            tokenizer.save_pretrained('best_minilm_tokenizer')
            print(f"   New best model saved! (Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f})")

    # Plot training history
    plot_training_history(train_losses, val_losses, train_accs, val_accs)

    # Load best model for threshold calibration + test evaluation
    print("\n Loading best model.")
    checkpoint = torch.load('best_minilm_nsfw_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])

    # --- Threshold calibration on the VALIDATION set (matches the other models) ---
    print("\n Calibrating decision threshold on the validation set.")
    _, _, val_labels_full, val_probs = evaluate(model, val_loader, criterion, device)
    best_threshold = find_optimal_threshold(val_probs, val_labels_full)

    # --- Final evaluation on the TEST set (apply calibrated threshold) ---
    print("\n Final evaluation on test set.")
    start_time = time.time()
    test_loss, _, test_labels, test_probs = evaluate(model, test_loader, criterion, device)
    end_time = time.time()

    # Apply the validation-calibrated threshold to the positive (NSFW) probability
    test_preds = (test_probs[:, 1] >= best_threshold).astype(int)

    # Efficiency metrics
    total_runtime = end_time - start_time
    num_samples = len(test_labels)
    samples_per_second = num_samples / total_runtime if total_runtime > 0 else 0
    latency_per_sample = (total_runtime / num_samples) * 1000 if num_samples > 0 else 0

    # Macro-averaged metrics -> directly comparable across all models
    accuracy = accuracy_score(test_labels, test_preds)
    precision_macro = precision_score(test_labels, test_preds, average='macro', zero_division=0)
    recall_macro = recall_score(test_labels, test_preds, average='macro', zero_division=0)
    f1_macro = f1_score(test_labels, test_preds, average='macro', zero_division=0)
    f1_weighted = f1_score(test_labels, test_preds, average='weighted', zero_division=0)
    # Positive-class (NSFW) metrics retained for reference
    precision_pos = precision_score(test_labels, test_preds, zero_division=0)
    recall_pos = recall_score(test_labels, test_preds, zero_division=0)
    f1_pos = f1_score(test_labels, test_preds, zero_division=0)
    cm = confusion_matrix(test_labels, test_preds)

    print("FINAL EVALUATION METRICS (TEST SET)")
    print(f"  Decision threshold (validation-calibrated): {best_threshold:.2f}")
    print(f"\n Primary metrics (macro-averaged -- comparable across all models):")
    print(f"  Accuracy:            {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Precision (Macro):   {precision_macro:.4f}")
    print(f"  Recall (Macro):      {recall_macro:.4f}")
    print(f"  F1-Score (Macro):    {f1_macro:.4f}")
    print(f"  F1-Score (Weighted): {f1_weighted:.4f}")
    print(f"\n Positive-class (NSFW) reference:")
    print(f"  Precision: {precision_pos:.4f} | Recall: {recall_pos:.4f} | F1: {f1_pos:.4f}")

    print(f"\n Efficiency Metrics:")
    print(f"  Samples per Second: {samples_per_second:.2f}")
    print(f"  Latency per Sample: {latency_per_sample:.4f} ms")
    print(f"  Total Runtime:      {total_runtime:.4f} seconds")

    print(f"\n Confusion Matrix:")
    print(f"{'':>15} {'Predicted SFW':>15} {'Predicted NSFW':>15}")
    print(f"{'Actual SFW':<15} {cm[0][0]:>15,} {cm[0][1]:>15,}")
    print(f"{'Actual NSFW':<15} {cm[1][0]:>15,} {cm[1][1]:>15,}")

    print("CLASSIFICATION REPORT")
    print(classification_report(test_labels, test_preds,
                                target_names=['SFW', 'NSFW'], digits=4))

    # Plot confusion matrix
    plot_confusion_matrix(cm)

    # Save deployment config (threshold + base) -- mirrors the other transformers
    deploy_cfg = {
        'decision_threshold': float(best_threshold),
        'max_length': max_length,
        'base_model': model_name,
        'num_labels': 2,
        'language_note': 'Multilingual MiniLM base (50+ langs incl. Filipino/Tagalog); largest base in study'
    }
    with open('minilm_deployment_config.json', 'w') as f:
        json.dump(deploy_cfg, f, indent=2)
    print(" Saved deployment config (threshold + max_length) to minilm_deployment_config.json")

    # Inference speed test
    print("INFERENCE SPEED TEST")
    test_text = "Test inference speed"
    num_runs = 100
    total_time = 0
    for _ in range(num_runs):
        _, _, inf_time = predict(test_text, model, tokenizer, device, max_length)
        total_time += inf_time
    avg_time = total_time / num_runs
    print(f"Average inference time: {avg_time:.2f} ms")
    print(f"Throughput: ~{1000/avg_time:.1f} predictions/second")

    # Inference examples (uses the calibrated threshold)
    print("INFERENCE EXAMPLES")
    test_examples = [
        "Magandang umaga po!",
        "Kumusta ka?",
        "Salamat sa tulong mo",
        "Puta ka",
        "Anong oras na?",
    ]
    for i, text in enumerate(test_examples, 1):
        _, probs, inf_time = predict(text, model, tokenizer, device, max_length)
        pred = int(probs[1] >= best_threshold)
        label = "NSFW" if pred == 1 else "SFW"
        confidence = probs[pred] * 100
        print(f"\n[{i}] Text: \"{text}\"")
        print(f"    Prediction: {label} (Confidence: {confidence:.2f}%, threshold {best_threshold:.2f})")
        print(f"    Probabilities: SFW={probs[0]:.4f}, NSFW={probs[1]:.4f}")
        print(f"    Inference time: {inf_time:.2f} ms")

    # Export Models
    print("\n EXPORTING MODELS")
    onnx_path = "exported_models/minilm_nsfw.onnx"
    export_to_onnx(model, tokenizer, onnx_path, device)
    export_to_tflite_ai_edge(model, model_name, "exported_models", max_length=max_length)

    print("TRAINING COMPLETE!")
    print(f" Best model saved: 'best_minilm_nsfw_model.pth'")
    print(f" Tokenizer saved: 'best_minilm_tokenizer/'")
    print(f" Deployment config: 'minilm_deployment_config.json'")
    print(f"\n Best validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")
    print(f" Final test accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f" Final test Macro-F1: {f1_macro:.4f} | Calibrated threshold: {best_threshold:.2f}")

    return model, tokenizer

In [17]:
# Run the training
if __name__ == "__main__":
    # Path to CSV file
    CSV_PATH = "cleaned_data.csv"

    # Training parameters optimized for A100 GPU
    EPOCHS = 10           # A100 is fast, sufficient epochs
    BATCH_SIZE = 128      # MiniLM-L12 is larger than TinyBERT; 128 fits comfortably on A100
    LEARNING_RATE = 5e-5  # Adjusted learning rate for larger batch size
    MAX_LENGTH = 128      # Maximum sequence length

    model, tokenizer = main(
        csv_path=CSV_PATH,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        max_length=MAX_LENGTH
    )

 Loading dataset.
  Transforming text labels to binary...
  Dataset loaded successfully!
  Total samples: 5,579
  Label distribution:
    - SFW (0): 3,079 (55.2%)
    - NSFW (1): 2,500 (44.8%)

 Splitting dataset (70/15/15, stratified).
  Training set: 3,905 samples
  Validation set: 837 samples
  Test set: 837 samples

 Initializing Multilingual MiniLM model.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

(00:13) [START] LiteRT-Torch Convert > Run LiteRT Converter Passes

(00:13) [ FAIL] LiteRT-Torch Convert > Run LiteRT Converter Passes

(00:13) [ FAIL] LiteRT-Torch Convert

  litert_torch INT8 conversion failed: Failed to run converter passes: /usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:134:0: error: 'stablehlo.uniform_dequantize' op operand #0 must be ranked tensor of per-tensor integer quantized or per-axis integer quantized values, but got 'tensor<384x384xi8>'

  Import in Android Studio via: File > New > Other > TensorFlow Lite Model.
TRAINING COMPLETE!
 Best model saved: 'best_minilm_nsfw_model.pth'
 Tokenizer saved: 'best_minilm_tokenizer/'
 Deployment config: 'minilm_deployment_config.json'

 Best validation accuracy: 0.9188 (91.88%)
 Final test accuracy: 0.9140 (91.40%)
 Final test Macro-F1: 0.9124 | Calibrated threshold: 0.50


Traceback (most recent call last):
  File "/tmp/ipykernel_1060/992121431.py", line 114, in export_to_tflite_ai_edge
    edge_q = aet.convert(
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/_convert/interface.py", line 323, in convert
    return Converter().convert(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/_convert/interface.py", line 208, in convert
    converted_model = core.convert_signatures(
                      ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/contextlib.py", line 81, in inner
    return func(*args, **kwds)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/_convert/core.py", line 158, in convert_signatures
    exporter = litert_converter.exported_programs_to_flatbuffer(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/_convert/litert_converter.py", line

In [18]:
# Zip the results folder
!zip -r Multilingual_MiniLM.zip /content/

  adding: content/ (stored 0%)
  adding: content/.config/ (stored 0%)
  adding: content/.config/gce (stored 0%)
  adding: content/.config/.last_survey_prompt.yaml (stored 0%)
  adding: content/.config/.last_opt_in_prompt.yaml (stored 0%)
  adding: content/.config/config_sentinel (stored 0%)
  adding: content/.config/configurations/ (stored 0%)
  adding: content/.config/configurations/config_default (deflated 15%)
  adding: content/.config/logs/ (stored 0%)
  adding: content/.config/logs/2026.06.04/ (stored 0%)
  adding: content/.config/logs/2026.06.04/13.39.22.644527.log (deflated 87%)
  adding: content/.config/logs/2026.06.04/13.39.37.138439.log (deflated 56%)
  adding: content/.config/logs/2026.06.04/13.39.36.079406.log (deflated 57%)
  adding: content/.config/logs/2026.06.04/13.39.24.274402.log (deflated 58%)
  adding: content/.config/logs/2026.06.04/13.39.12.467650.log (deflated 58%)
  adding: content/.config/logs/2026.06.04/13.38.54.010366.log (deflated 92%)
  adding: content/.con

In [19]:
from google.colab import files

files.download('Multilingual_MiniLM.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>